apply trasforms and generate efficient dataloader for training, validation, and testing

Trainer steps:
- Generage vocabulary (generate purely from training data, use "missing" for unseen values in val and test)
- Embeddings for sparse data
- Log or quantile-based transforms for continuous (or discrete) data

About Criteo Attribution dataset:
- This dataset represent a sample of 30 days of Criteo live traffic data. 
- Each line corresponds to one impression (a banner) that was displayed to a user.
- "the last 7 days of log are considered for testing and for each test day we use the 21 previous days for learning" paper [8] section 4.3.2

Features:
- timestamp
	- use indirectly
		- data split (e.g. use first 3 weeks for training, last week for validation and test)
	- historical features
		- last n campaigns a user clicked
		- last n conversions
- uid
	- learnable user embedding
	- Meta paper [6]. use a two tower DLRM for user and ad features
    - Why DLRM? similar implicit/explicit feature dichotomy as DCN (bounded to 2nd order), with less learnable parameters. DCN-like strucuture is worth exploring if latency and recousce costs would allow it. With DLRM we explore all combinations of dense and sparse feature crosses (2nd order only) that the model can learn from, simpler than manually hand engineering feature like in Wide and deep paper [5].
- campaign id
	- learnable ad embedding
- conversion - objective task 1, satisfaction
- click - objective task 2, engagement
- click_pos
	- will be used to train shallow network to learn selection bias, ads at higher positions are biased to be clicked by users, but the position does not necessarily represent the user utility. The model needs to learn this distinction, which will be done by building a dedicated shallow network to learn this bias, according to paper [3]
- click_nb
	- data leakage, if positive then it means conversion, something we will not know ahead of time
- cat
    - anonymous ad related features
	- include all, create a vocabulary, then create embedding tables for each
- cost
	- not used for training, but can be used to approximate the total value and total value divergence metrics
	- cost is typically lower than bid due to the second-price strategy, where an advertiser pays the second highest bid, instead of theirs
- Excluded Attribution related features
	- coversion_timestamp
	- conversion_id
	- attribution

In [2]:
from datasets import load_dataset
import os

train_file_path = "../../datasets/criteo_attribution_dataset"

full = load_dataset(
    'csv',
    data_files=os.path.join(train_file_path, "criteo_attribution_dataset.tsv"),
    delimiter='\t'
)
full

/Users/paataugrekhelidze/Projects/Recsys/ranking/ads_Moe/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id', 'attribution', 'click', 'click_pos', 'click_nb', 'cost', 'cpo', 'time_since_last_click', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9'],
        num_rows: 16468027
    })
})

In [4]:
from sklearn.preprocessing import OrdinalEncoder

# list features that will require embeddings and print their cardinality
sparse_features = ["uid", "campaign"]+[f"cat{i}" for i in range(1, 10)]
sparse_voc_size = list()
V = dict()
for f in sparse_features:
    unique_values = full["train"].unique(f)
    
    
    print(f, len(unique_values))
    sparse_voc_size.append(len(unique_values))
    V[f] = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=len(unique_values)+1, dtype=int)
    V[f].fit([[val] for val in unique_values])

    # separate indices for missing and unknown embeddings?
print(sparse_features)


uid 6142256
campaign 675
cat1 9
cat2 70
cat3 1829
cat4 21
cat5 51
cat6 30
cat7 57196
cat8 11
cat9 30
['uid', 'campaign', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']


Create a history of last n campaign interactions per user (clicks, conversions)

In [13]:
df = full["train"].to_polars()

In [14]:
import polars as pl

n = 2

def get_history(df, event_col, output_name):
    # Step A: Get ONLY the rows where the event happened
    history_events = (
        df.filter(pl.col(event_col) == 1)
        .select(["uid", "timestamp", "campaign"])
        # Group by user and collect ALL their campaigns into one list
        .group_by("uid")
        .agg([
            pl.col("timestamp"),
            pl.col("campaign")
        ])
    )

    # Step B: Join the lists back to every row in the original data
    return (
        df.join(history_events, on="uid", how="left")
        .with_columns([
            # For every row, filter the user's total history to only 
            # campaigns that happened BEFORE the current row's timestamp
            pl.struct(["timestamp", "timestamp_right", "campaign_right"])
            .map_elements(
                lambda x: (
                    list(dict.fromkeys([
                        camp for ts, camp in zip(x["timestamp_right"], x["campaign_right"]) 
                        if ts < x["timestamp"] # this could alternitavely be limitied to last few days
                    ]))[-n:] 
                    if x["timestamp_right"] is not None and x["campaign_right"] is not None
                    else []
                ),
                return_dtype=pl.List(pl.Int64) # return int ids
            )
            .alias(output_name)
        ])
        .drop(["timestamp_right", "campaign_right"])
    )
df = get_history(df, "click", "last_n_click_campaigns")
df = get_history(df, "conversion", "last_n_conversion_campaigns")

In [15]:
# test by viewing a uid with lots of clicks
group_counts = df.filter(pl.col("click") == 1).group_by("uid").len()
largest_group_info = group_counts.sort("len", descending=True).head(1)
active_uid, active_uid_count = largest_group_info["uid"].item(), largest_group_info["len"].item()
print(active_uid, active_uid_count)

8826511 849


In [16]:
df.filter(pl.col("uid") == active_uid)["timestamp", "uid", "campaign", "click"]

timestamp,uid,campaign,click
i64,i64,i64,i64
23326,8826511,13576413,1
23331,8826511,13576413,1
23552,8826511,11387831,1
23567,8826511,11387831,1
23578,8826511,13576413,1
…,…,…,…
2533057,8826511,11387831,1
2533545,8826511,11387831,0
2533809,8826511,11387831,1


In [17]:
df.filter(pl.col("uid") == active_uid)["timestamp", "uid", "last_n_click_campaigns"].show(10)

timestamp,uid,last_n_click_campaigns
i64,i64,list[i64]
23326,8826511,[]
23331,8826511,[13576413]
23552,8826511,[13576413]
23567,8826511,"[13576413, 11387831]"
23578,8826511,"[13576413, 11387831]"
23622,8826511,"[13576413, 11387831]"
26826,8826511,"[13576413, 11387831]"
26945,8826511,"[13576413, 11387831]"
27349,8826511,"[13576413, 11387831]"


In [18]:
# before sparse feature transformation
df[sparse_features].head(10)

uid,campaign,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
20073966,22589171,5824233,9312274,3490278,29196072,11409686,1973606,25162884,29196072,29196072
24607497,884761,30763035,9312274,14584482,29196072,11409686,1973606,22644417,9312274,21091111
28474333,18975823,138937,9312274,10769841,29196072,5824237,138937,1795451,29196072,15351056
7306395,29427842,28928366,26597095,12435261,23549932,5824237,1973606,9180723,29841067,29196072
25357769,13365547,138937,26597094,31616034,29196072,11409684,26597096,4480345,29196072,29196072
93907,17686799,30763035,9068207,9107790,29196072,32440044,1973606,2687461,29841067,21091108
19923387,31772643,30763035,9312274,5028397,29196072,32440044,32440041,14074087,29196072,21091108
28451570,20843295,138937,9312274,15403272,29196072,32440042,28928366,8556462,29196072,29196072
5588915,27491436,138937,9312274,4281154,29196072,28928366,29196072,21857352,29196072,29196072


In [19]:
# Transform all sparse features using the vocabularies
for feature in sparse_features:
    # Get the column as numpy array, reshape for sklearn, transform, flatten
    encoded_values = V[feature].transform(df[feature].to_numpy().reshape(-1, 1)).flatten()
    # Update the column in the dataframe
    df = df.with_columns([
        pl.Series(feature, encoded_values, dtype=pl.Int64)
    ])

df[sparse_features].head(10)

uid,campaign,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
3798903,450,3,23,217,17,16,1,44488,8,25
4656463,18,8,23,876,17,16,1,40052,3,19
5387918,399,0,23,671,17,7,0,3042,8,11
1382198,598,7,46,748,14,7,1,16035,9,25
4798501,279,0,45,1779,17,14,19,7727,8,25
17886,379,8,17,584,17,47,1,4572,9,18
3770345,657,8,23,362,17,47,26,24737,8,18
5383670,420,0,23,919,17,46,22,14914,8,25
1057756,557,0,23,297,17,41,23,38762,8,25


In [ ]:
# unfortunately, majority of position data is not available, which might prevent us from learning position bias
# we will treat -1 as missing value, which means that during inference (test data) we will also set all position values to "-1"
# 3 ways to treat the missing value
# Option 1: set position_0 = 1 for all impressions with missing value, not ideal, we have training data with the actual position_0
# Option 2: set 0 to all positions, the final output will be miscalibrated, not ideal.
# Option 3 (my pick): set equal probability for all positions that add up to 1, every position will take part in the computation. 
pos_group_counts = df.group_by("click_pos").len()
sorted_pos_group = pos_group_counts.sort("len", descending=True)
sorted_pos_group.show(10)


click_pos,len
i64,u32
-1,15661831
0,400152
1,139933
2,73191
3,44990
4,30527
5,21932
6,16275
7,12460


In [21]:
# create one_hot encoding for click_position
from sklearn.preprocessing import OneHotEncoder

unique_values = full["train"].unique("click_pos")
V["click_pos"] = OneHotEncoder(sparse_output=True)
V["click_pos"].fit([[val] for val in unique_values])


click_pos_encoded = V["click_pos"].transform(df["click_pos"].to_numpy().reshape(-1, 1))

In [22]:
# Create separate columns for each position
n_positions = click_pos_encoded.shape[1]
position_df = pl.DataFrame(
    {f"position_{i-1}": click_pos_encoded[:, i].toarray().flatten()
     for i in range(1, n_positions)} # skip click_pos = -1
)

df = pl.concat([df, position_df], how="horizontal")

In [23]:
n_positions

175

In [24]:
# filter by click_pos = -1, set position values to an equal amount
equal_prob = 1 / (n_positions-1)
df = df.with_columns([
    pl.when(pl.col("click_pos") == -1)
    .then(pl.lit(equal_prob))
    .otherwise(pl.col(f"position_{i-1}"))
    .alias(f"position_{i-1}")
    for i in range(1, n_positions)
])
df.head()

timestamp,uid,campaign,conversion,conversion_timestamp,conversion_id,attribution,click,click_pos,click_nb,cost,cpo,time_since_last_click,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,last_n_click_campaigns,last_n_conversion_campaigns,position_0,position_1,position_2,position_3,position_4,position_5,position_6,position_7,position_8,position_9,position_10,position_11,position_12,…,position_137,position_138,position_139,position_140,position_141,position_142,position_143,position_144,position_145,position_146,position_147,position_148,position_149,position_150,position_151,position_152,position_153,position_154,position_155,position_156,position_157,position_158,position_159,position_160,position_161,position_162,position_163,position_164,position_165,position_166,position_167,position_168,position_169,position_170,position_171,position_172,position_173
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,list[i64],list[i64],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,3798903,450,0,-1,-1,0,0,-1,-1,0.00001,0.390794,-1,3,23,217,17,16,1,44488,8,25,[],[],0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,…,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747
2,4656463,18,0,-1,-1,0,0,-1,-1,0.00001,0.0596,423858,8,23,876,17,16,1,40052,3,19,[],[],0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,…,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747
2,5387918,399,0,-1,-1,0,0,-1,-1,0.000183,0.149706,8879,0,23,671,17,7,0,3042,8,11,[],[],0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,…,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747
3,1382198,598,1,1449193,3063962,0,1,0,7,0.000094,0.154785,-1,7,46,748,14,7,1,16035,9,25,[],[],1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4798501,279,0,-1,-1,0,0,-1,-1,0.000032,0.037583,-1,0,45,1779,17,14,19,7727,8,25,[],[],0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,…,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747


In [25]:
# save processed data
df = df[["timestamp"] + sparse_features + ["last_n_click_campaigns", "last_n_conversion_campaigns"] + [f"position_{i-1}" for i in range(1, n_positions)] + ["cost", "click", "conversion"]]
df.head()


timestamp,uid,campaign,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,last_n_click_campaigns,last_n_conversion_campaigns,position_0,position_1,position_2,position_3,position_4,position_5,position_6,position_7,position_8,position_9,position_10,position_11,position_12,position_13,position_14,position_15,position_16,position_17,position_18,position_19,position_20,position_21,position_22,…,position_140,position_141,position_142,position_143,position_144,position_145,position_146,position_147,position_148,position_149,position_150,position_151,position_152,position_153,position_154,position_155,position_156,position_157,position_158,position_159,position_160,position_161,position_162,position_163,position_164,position_165,position_166,position_167,position_168,position_169,position_170,position_171,position_172,position_173,cost,click,conversion
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,list[i64],list[i64],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
0,3798903,450,3,23,217,17,16,1,44488,8,25,[],[],0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,…,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.00001,0,0
2,4656463,18,8,23,876,17,16,1,40052,3,19,[],[],0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,…,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.00001,0,0
2,5387918,399,0,23,671,17,7,0,3042,8,11,[],[],0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,…,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.000183,0,0
3,1382198,598,7,46,748,14,7,1,16035,9,25,[],[],1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000094,1,1
3,4798501,279,0,45,1779,17,14,19,7727,8,25,[],[],0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,…,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.005747,0.000032,0,0


In [26]:
max_timestamp = df.select(pl.col("timestamp").max()).item()
DAYS_IN_SECONDS = 7 * 86400 # 7 days
split_timestamp = max_timestamp - DAYS_IN_SECONDS


# Split into train and validation
train_df = df.filter(pl.col("timestamp") < split_timestamp)
val_df = df.filter(pl.col("timestamp") >= split_timestamp)

print(f"Train samples: {len(train_df)}, Validation samples: {len(val_df)}")

Train samples: 12976588, Validation samples: 3491439


In [27]:
# ablation of position for validatin data
# Set all positions to equal probability (simulating missing position at inference)
equal_prob = 1 / (n_positions-1)
val_df = val_df.with_columns([
    pl.lit(equal_prob)
    .alias(f"position_{i-1}")
    for i in range(1, n_positions)
])

In [28]:
filepath = train_file_path + '/processed'
for ds in ["train", "val"]:
    path = os.path.join(filepath, ds)
    if not os.path.exists(path):
        os.makedirs(path)

# Convert to HuggingFace Datasets
train_df.write_parquet(os.path.join(filepath, "train/train.parquet"))
val_df.write_parquet(os.path.join(filepath, "val/val.parquet"))

print(f"Saved train dataset to {os.path.join(filepath, 'train/train.parquet')}")
print(f"Saved validation dataset to {os.path.join(filepath, 'val/val.parquet')}")

Saved train dataset to ../../datasets/criteo_attribution_dataset/processed/train/train.parquet
Saved validation dataset to ../../datasets/criteo_attribution_dataset/processed/val/val.parquet


In [1]:
# test dataloader
from datasets import Dataset
import os

filepath = "../../datasets/criteo_attribution_dataset/processed"

train_df = Dataset.from_parquet(os.path.join(filepath, 'train/train.parquet'))
train_df

/Users/paataugrekhelidze/Projects/Recsys/ranking/ads_Moe/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['timestamp', 'uid', 'campaign', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9', 'last_n_click_campaigns', 'last_n_conversion_campaigns', 'position_0', 'position_1', 'position_2', 'position_3', 'position_4', 'position_5', 'position_6', 'position_7', 'position_8', 'position_9', 'position_10', 'position_11', 'position_12', 'position_13', 'position_14', 'position_15', 'position_16', 'position_17', 'position_18', 'position_19', 'position_20', 'position_21', 'position_22', 'position_23', 'position_24', 'position_25', 'position_26', 'position_27', 'position_28', 'position_29', 'position_30', 'position_31', 'position_32', 'position_33', 'position_34', 'position_35', 'position_36', 'position_37', 'position_38', 'position_39', 'position_40', 'position_41', 'position_42', 'position_43', 'position_44', 'position_45', 'position_46', 'position_47', 'position_48', 'position_49', 'position_50', 'position_51', 'position_52', 'position_53', 'position_54',

In [2]:
len(train_df)

12976588

In [7]:
from torch.utils.data import DataLoader
from Ads import AdsDataset
from functools import partial

B = 2048
WORKERS = 0

# Define voc size for user and campaign ids
user_v = 2e6
campaign_v = 400


# Create a partial function with the arguments bound
collate_with_args = partial(AdsDataset._collate_batch, user_v=user_v, campaign_v=campaign_v)


train_dataset = AdsDataset(data= train_df)
train_ld = DataLoader(dataset=train_dataset, batch_size=B, shuffle=False, num_workers=WORKERS, collate_fn=collate_with_args)
total = len(train_df) // B
# for ix, sample in enumerate(train_ld):
#     print(f"[{ix}/{total}]")

treat missing history with a separate unique id with (learnable embedding), let the model learn what the embedding should be for missing history instead of forcing a certain behavior with 0 values.

But why not? because we are forcing a model to take a value 0 for a missing history, even though we do not know what the 0 embedding really represents in the latent space.

If we had history for most of the data, then we would use a 5-10% dropout to avoid overreliance on this feature, especially since it might be missing for new or inactive users, and the model needs to learn to be efficient without the history embedding. But, since most impressions have no history, this may not be necessary for our use case.

The same dropout strategy can be incorporated for the selection bias.

In [8]:
iterator = iter(train_ld)
sample = next(iterator)
for k in sample.keys():
    print(k, sample[k].shape)

uid torch.Size([2048])
campaign torch.Size([2048])
cat1 torch.Size([2048])
cat2 torch.Size([2048])
cat3 torch.Size([2048])
cat4 torch.Size([2048])
cat5 torch.Size([2048])
cat6 torch.Size([2048])
cat7 torch.Size([2048])
cat8 torch.Size([2048])
cat9 torch.Size([2048])
position_0 torch.Size([2048])
position_1 torch.Size([2048])
position_2 torch.Size([2048])
position_3 torch.Size([2048])
position_4 torch.Size([2048])
position_5 torch.Size([2048])
position_6 torch.Size([2048])
position_7 torch.Size([2048])
position_8 torch.Size([2048])
position_9 torch.Size([2048])
position_10 torch.Size([2048])
position_11 torch.Size([2048])
position_12 torch.Size([2048])
position_13 torch.Size([2048])
position_14 torch.Size([2048])
position_15 torch.Size([2048])
position_16 torch.Size([2048])
position_17 torch.Size([2048])
position_18 torch.Size([2048])
position_19 torch.Size([2048])
position_20 torch.Size([2048])
position_21 torch.Size([2048])
position_22 torch.Size([2048])
position_23 torch.Size([2048])

In [22]:
sample = next(iterator)
sample["last_n_conversion_campaigns_1D"].shape

torch.Size([2049])

In [23]:
import torch

unique_values, counts = torch.unique(sample["last_n_conversion_campaigns_1D"], return_counts=True)

print("Unique values:", unique_values)
print("Counts:", counts)

Unique values: tensor([ 43, 106, 111, 118, 132, 170, 184, 240, 243, 244, 301, 304, 307, 359,
        381, 382, 383, 393, 400])
Counts: tensor([   2,    1,    3,    1,    1,    1,    1,    1,    1,    1,    6,    4,
           2,    1,    2,    4,    1,    3, 2013])


2049